# Mapeando Datos y Fijando Valores
## Dominando Diccionarios y Tuplas en Python

**Semana 09 - Fundamentos de Python**  
**Cuaderno de trabajo del estudiante**  
**Caso:** análisis de hasta 1,000,000 de pacientes sintéticos

En esta práctica trabajamos con diccionarios para organizar conteos y con tuplas para conservar resultados relacionados. Comenzamos con una muestra y, cuando el análisis esté listo, usamos el mismo código con el millón de registros.

Al finalizar podremos:

1. recorrer claves y valores de un diccionario;
2. contar categorías con `.get()`;
3. recorrer una lista interna con ciclos anidados;
4. construir un diccionario que contiene otros diccionarios;
5. representar un resultado mediante una tupla.


## Preparamos y cargamos los datos

El archivo contiene datos clínicos sintéticos. Los nombres iniciales permiten reconocer el conjunto, pero las edades, enfermedades, provincias, medicamentos y síntomas no representan información real.

Iniciamos con una muestra independiente de 5,000 registros. En la Actividad 8 cambiaremos el control para generar y cargar el millón. No abrimos los archivos JSON directamente en el editor.


In [1]:
from pathlib import Path
import sys

carpeta_semana = Path.cwd()
if not (carpeta_semana / "datos_s09.py").exists():
    carpeta_semana = Path.cwd() / "Semana 09"

if not (carpeta_semana / "datos_s09.py").exists():
    raise FileNotFoundError("Abrimos el notebook desde la raíz del curso o desde Semana 09.")

sys.path.insert(0, str(carpeta_semana.resolve()))
from datos_s09 import cargar_datos


In [2]:
USAR_MILLON = False
registros, ruta_datos = cargar_datos(USAR_MILLON, carpeta_semana)

print(f"Registros cargados: {len(registros):,}")
print(f"Archivo utilizado: {ruta_datos.name}")


Progreso: 500 de 5,000 pacientes (10%)
Progreso: 1,000 de 5,000 pacientes (20%)
Progreso: 1,500 de 5,000 pacientes (30%)
Progreso: 2,000 de 5,000 pacientes (40%)
Progreso: 2,500 de 5,000 pacientes (50%)
Progreso: 3,000 de 5,000 pacientes (60%)
Progreso: 3,500 de 5,000 pacientes (70%)
Progreso: 4,000 de 5,000 pacientes (80%)
Progreso: 4,500 de 5,000 pacientes (90%)
Progreso: 5,000 de 5,000 pacientes (100%)
Registros cargados: 5,000
Archivo utilizado: clinica_s09_muestra.json


## Iniciamos la cadena de análisis

Partimos de `registros`, el conjunto seleccionado en la celda anterior. Cada actividad utilizará resultados de la actividad previa hasta construir un resumen por provincia.


In [55]:
from verificaciones_s09 import (
    verificar_contador, verificar_contador_filtrado, verificar_mapa_anidado,
    verificar_inicio_mapa, verificar_maximo, verificar_totales_por_grupo,
    verificar_tupla,
)

print("Verificaciones de apoyo disponibles.")
print("  - verificar_contador")
print("  - verificar_contador_filtrado")
print("  - verificar_mapa_anidado")
print("  - verificar_inicio_mapa")
print("  - verificar_maximo")
print("  - verificar_totales_por_grupo")
print("  - verificar_tupla")
print("Ejemplo de uso: verificar_contador(contador, 'nombre_del_contador')")
print("Recuerde que los contadores y mapas deben ser diccionarios de Python.")

Verificaciones de apoyo disponibles.
  - verificar_contador
  - verificar_contador_filtrado
  - verificar_mapa_anidado
  - verificar_inicio_mapa
  - verificar_maximo
  - verificar_totales_por_grupo
  - verificar_tupla
Ejemplo de uso: verificar_contador(contador, 'nombre_del_contador')
Recuerde que los contadores y mapas deben ser diccionarios de Python.


In [ ]:
print("Comenzamos con el primer registro del conjunto seleccionado.")


## Actividad 1 - Exploramos un registro

Seleccionamos el primer paciente de `registros`. Consultamos su enfermedad, provincia y lista de síntomas; estos valores se usarán en las actividades siguientes.


In [19]:
# Seleccionamos el primer registro y exploramos sus tipos, claves y campos.
print("Primer registro:")
print(registros[0]) 
print("Tipo del registro:", type(registros[0]))
print("Claves del registro:", list(registros[0].keys()))
print("Campos del registro:")
for clave, valor in registros[0].items():
    print(f"  {clave}: {valor} (tipo: {type(valor)})")  

Primer registro:
{'carnet': 115000001, 'nombre': 'Germán Antonio Cerdas Valle', 'edad': 22, 'genero': 'M', 'provincia': 'Alajuela', 'canton': 'Alajuela', 'visitas': 22, 'enfermedad': 'alzheimer', 'medicamento': 'losartán', 'activo': True, 'sintomas': ['náuseas', 'fatiga', 'dolor muscular']}
Tipo del registro: <class 'dict'>
Claves del registro: ['carnet', 'nombre', 'edad', 'genero', 'provincia', 'canton', 'visitas', 'enfermedad', 'medicamento', 'activo', 'sintomas']
Campos del registro:
  carnet: 115000001 (tipo: <class 'int'>)
  nombre: Germán Antonio Cerdas Valle (tipo: <class 'str'>)
  edad: 22 (tipo: <class 'int'>)
  genero: M (tipo: <class 'str'>)
  provincia: Alajuela (tipo: <class 'str'>)
  canton: Alajuela (tipo: <class 'str'>)
  visitas: 22 (tipo: <class 'int'>)
  enfermedad: alzheimer (tipo: <class 'str'>)
  medicamento: losartán (tipo: <class 'str'>)
  activo: True (tipo: <class 'bool'>)
  sintomas: ['náuseas', 'fatiga', 'dolor muscular'] (tipo: <class 'list'>)


## Actividad 2 - Convertimos campos en pares y tuplas

Partimos de `primer_paciente`. Creamos `datos_observados` con su enfermedad, provincia y síntomas. Recorremos estos campos con `.items()`, formamos una tupla con la enfermedad y la desempaquetamos para obtener `enfermedad_observada`.


In [48]:
# Creamos datos_observados y mostramos cada asociación al recorrer .items().
# Formamos par_observado con la enfermedad y obtenemos enfermedad_observada.
print("Exploramos el primer registro y formamos un par observado.")
datos_observados = registros[0]
for campo_observado, enfermedad_observada in datos_observados.items():
    par_observado = (campo_observado, enfermedad_observada)
    print(f"  Par observado: {par_observado}")
    print(f"    Campo observado: {campo_observado}")  
    print(f"    Enfermedad observada: {enfermedad_observada}")
    print(f"    Tipo de campo observado: {type(campo_observado)}")
    print(f"    Tipo de enfermedad observada: {type(enfermedad_observada)}") 
    print(f"    Tipo de par observado: {type(par_observado)}") 
    print(f"    Tipo de datos_observados: {type(datos_observados)}") 
    print(f"    Tipo de registros: {type(registros)}")  
    for clave, valor in registros[0].items():
         print(f"    Clave: {clave}, Valor: {valor}, Tipo de valor: {type(valor)}") 
         print(f"    Tipo de clave: {type(clave)}") 


Exploramos el primer registro y formamos un par observado.
  Par observado: ('carnet', 115000001)
    Campo observado: carnet
    Enfermedad observada: 115000001
    Tipo de campo observado: <class 'str'>
    Tipo de enfermedad observada: <class 'int'>
    Tipo de par observado: <class 'tuple'>
    Tipo de datos_observados: <class 'dict'>
    Tipo de registros: <class 'list'>
    Clave: carnet, Valor: 115000001, Tipo de valor: <class 'int'>
    Tipo de clave: <class 'str'>
    Clave: nombre, Valor: Germán Antonio Cerdas Valle, Tipo de valor: <class 'str'>
    Tipo de clave: <class 'str'>
    Clave: edad, Valor: 22, Tipo de valor: <class 'int'>
    Tipo de clave: <class 'str'>
    Clave: genero, Valor: M, Tipo de valor: <class 'str'>
    Tipo de clave: <class 'str'>
    Clave: provincia, Valor: Alajuela, Tipo de valor: <class 'str'>
    Tipo de clave: <class 'str'>
    Clave: canton, Valor: Alajuela, Tipo de valor: <class 'str'>
    Tipo de clave: <class 'str'>
    Clave: visitas, Valor

## Actividad 3 - Construimos un conteo pequeño

Partimos de la enfermedad observada y recorremos solamente `registros[:5]`. Construimos manualmente `conteo_pequeno_enfermedades` para representar cuántas veces aparece cada enfermedad en esos cinco registros.

Usamos `diccionario.get(clave, valor_predeterminado)` para consultar una clave sin producir un error cuando todavía no existe. Si la clave está presente, obtenemos su valor actual; si no está presente, obtenemos el valor predeterminado. En un conteo utilizamos `0` como punto de partida.

Podemos actualizar una categoría con este patrón:

```python
conteo[valor] = conteo.get(valor, 0) + 1
```


In [ ]:
conteo_pequeno_enfermedades = {}

# Recorremos los primeros cinco registros y actualizamos el conteo manual.

print(
    "Enfermedad observada en el conteo:", enfermedad_observada,
    conteo_pequeno_enfermedades.get(enfermedad_observada, 0),
)
verificar_contador(
    conteo_pequeno_enfermedades, registros[:5], "enfermedad", "conteo pequeño"
)


## Actividad 4 - Generalizamos el conteo en una función

Convertimos el patrón de la Actividad 3 en nuestra única función obligatoria: `contar_por_campo(registros, campo)`. La aplicamos a enfermedad, provincia y medicamento, y consultamos una categoría real de cada conteo.


In [ ]:
def contar_por_campo(registros, campo):
    """Cuenta las apariciones de cada valor de un campo."""
    # Generalizamos aquí el patrón del conteo pequeño.
    pass

# Creamos los tres conteos y consultamos una clave existente de cada uno.

verificar_contador(conteo_enfermedades, registros, "enfermedad", "enfermedades")
verificar_contador(conteo_provincias, registros, "provincia", "provincias")
verificar_contador(conteo_medicamentos, registros, "medicamento", "medicamentos")


## Actividad 5 - Contamos síntomas con dos ciclos

Extendemos el patrón de conteo a la lista `sintomas` observada en la Actividad 1. Construimos `conteo_sintomas` con un ciclo para los registros y otro para sus síntomas.


In [ ]:
conteo_sintomas = {}

# Escribimos aquí los dos ciclos y actualizamos el conteo.

verificar_contador(
    conteo_sintomas,
    registros,
    "sintomas",
    "síntomas",
    campo_lista=True,
)


## Actividad 6 - Filtramos síntomas de una provincia

Partimos de la provincia del primer paciente y de los dos ciclos anteriores. Construimos `conteo_sintomas_provincia`, un diccionario plano que incluye únicamente los síntomas de esa provincia.


In [ ]:
provincia_filtrada = primer_paciente["provincia"]
conteo_sintomas_provincia = {}

# Filtramos por provincia y contamos sus síntomas con dos ciclos.

print("Total global de síntomas:", sum(conteo_sintomas.values()))
print("Total de la provincia:", sum(conteo_sintomas_provincia.values()))
verificar_contador_filtrado(
    conteo_sintomas_provincia, registros, "provincia",
    provincia_filtrada, "sintomas", "síntomas de la provincia",
)


## Actividad 7 - Construimos y consultamos el mapa provincial

Convertimos el filtro de la Actividad 6 en un mapa que conserva los conteos de todas las provincias. Primero creamos un checkpoint con el primer registro y después extendemos el mapa al resto de `registros`:

```python
{provincia: {síntoma: cantidad}}
```

Finalmente recorremos ambos niveles con `.items()`, totalizamos cada provincia y fijamos como tupla el síntoma mayor de `provincia_filtrada`.


### 7A - Iniciamos el mapa con un registro

Partimos de `primer_paciente`. Creamos el primer diccionario provincial con sus tres síntomas y comprobamos que el mapa represente únicamente este registro.


In [ ]:
sintomas_por_provincia = {}

# Iniciamos aquí el mapa con la provincia y los síntomas del primer registro.

verificar_inicio_mapa(
    sintomas_por_provincia, primer_paciente, "provincia", "sintomas",
    "el mapa inicial",
)


### 7B - Extendemos el mapa a todos los registros

Partimos del mapa inicial. Recorremos `registros[1:]`, agregamos las provincias que todavía no existen y actualizamos sus conteos de síntomas.


In [ ]:
# Extendemos aquí sintomas_por_provincia con los registros restantes.

assert (
    sintomas_por_provincia[provincia_filtrada]
    == conteo_sintomas_provincia
), "Comprobamos que el grupo provincial coincida con el conteo filtrado de la Actividad 6."
print("Comprobamos que el mapa conserva exactamente el conteo filtrado de la Actividad 6.")

verificar_mapa_anidado(
    sintomas_por_provincia,
    registros,
    "provincia",
    "sintomas",
    "síntomas por provincia (checkpoint completo)",
    campo_valor_lista=True,
)


### 7C - Consultamos los dos niveles del mapa

Recorremos el nivel de provincias y el nivel de síntomas mediante `.items()`. Construimos los totales provinciales y fijamos en una tupla el síntoma mayor de `provincia_filtrada`.


In [ ]:
totales_sintomas_por_provincia = {}
sintoma_mayor_provincia = ("", -1)

# Recorremos ambos niveles, totalizamos y actualizamos la tupla máxima.

verificar_totales_por_grupo(
    totales_sintomas_por_provincia, sintomas_por_provincia, "totales provinciales"
)
verificar_maximo(
    sintoma_mayor_provincia, sintomas_por_provincia[provincia_filtrada],
    "síntoma mayor de la provincia filtrada",
)


## Actividad 8 - Validamos con el millón y concluimos

Si `USAR_MILLON` continúa en `False`, nuestras conclusiones corresponden a la muestra. Para validar con el millón, cambiamos el valor a `True` en **Preparamos y cargamos los datos** y volvemos a ejecutar desde esa celda hasta la Actividad 7.

Escribimos exactamente tres conclusiones: cómo generalizamos el conteo pequeño, cómo el filtro provincial se convirtió en un mapa y cómo consultamos el resumen con el volumen seleccionado.


In [ ]:
conclusion_generalizacion = ""
conclusion_mapa_provincial = ""
conclusion_consulta_millon = ""

# Escribimos tres conclusiones con nuestras propias palabras.

if USAR_MILLON:
    print("Validamos la cadena con 1,000,000 de registros.")
else:
    print("Validamos la cadena con la muestra. Cambiamos USAR_MILLON para validar el millón.")

print(conclusion_generalizacion)
print(conclusion_mapa_provincial)
print(conclusion_consulta_millon)


## Cierre

Comprobamos las estructuras con el conjunto seleccionado. Nuestros diccionarios convierten los registros en conteos consultables y nuestras tuplas mantienen unidos los valores de un resultado.
